# Create Carlsbergfondet Awards (GRANT PATTERN, official sitemap/detail scrape)

Ingest of grants published by Carlsbergfondet on its official `carlsbergfondet.dk` "What we have funded" pages. The public English sitemap exposes 3,164 grant-detail URLs under `/en/what-we-have-funded/{slug}/`, and each detail page exposes labeled fields for title, applicant, institution, amount, year, and type of grant.

**Awarding body:** Carlsbergfondet - `F4320321504` (DK, ROR `https://ror.org/01kpjmx04`, DOI `10.13039/501100002808`).

**Schema choices:**
- One row per official grant detail page. The URL slug is the source-stable key and becomes `funder_award_id` with a `carlsberg-fondet-` prefix.
- `funding_type = 'grant'`.
- `funder_scheme = type_of_grant` from the labeled page field (for example `Semper Ardens: Accelerate`, `Research Infrastructure`, `Field Trips / Research Stays < 100,000`).
- `amount` and `currency` come from the page's per-grant `Amount` label. Amounts are parsed from strings such as `DKK 327,181`.
- `lead_investigator` is the page's `Name of applicant`; `affiliation.name` is the page's `Institution`; `affiliation.country` remains NULL because the page does not publish a country field.
- `description` remains NULL because the detail template does not expose a separate abstract/summary beyond the award title.
- No S3 or Databricks write was run by the contractor; admin must upload the parquet and execute this notebook.

**Source:** `https://www.carlsbergfondet.dk/en/sitemap.xml` plus per-grant detail pages. Source method: official source ladder 1.

**Local validation before handoff:** `scripts/local/carlsberg_fondet_to_s3.py --limit 10 --skip-upload` produced 10/10 coverage for title, applicant, amount, year, type, and URL. Full local crawl completed on 2026-05-26: 3,163 shipped rows from 3,164 grant-detail URLs after excluding one official sitemap test page (`cf22-0030`, title `TEST FORBINDELSE DRIFT`, amount `DKK 0`); 100% coverage for title, applicant, amount, year, type, and landing URL; parquet written locally with upload skipped for admin handoff.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carlsberg_fondet_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/carlsberg_fondet/carlsberg_fondet_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS raw_rows FROM openalex.awards.carlsberg_fondet_raw;

## Step 1.5: Inspect raw data, money shape, and key uniqueness

Expected source columns: `funder_award_id`, `slug`, `display_name`, `description`, `applicant_name`, `given_name`, `family_name`, `institution`, `amount`, `currency`, `amount_raw`, `year`, `start_date`, `end_date`, `type_of_grant`, `landing_page_url`, `source_url`, `http_status`, `year_raw`, `downloaded_at`, `declined`. All columns are written as STRING at the parquet write site per the runbook.

In [ ]:
%sql
DESCRIBE openalex.awards.carlsberg_fondet_raw;

In [ ]:
%sql
SELECT *
FROM openalex.awards.carlsberg_fondet_raw
LIMIT 10;

In [ ]:
%sql
-- Key uniqueness: must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.carlsberg_fondet_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id
LIMIT 20;

In [ ]:
%sql
-- Required-field coverage from the official detail pages.
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(applicant_name) AS has_applicant,
    COUNT(institution) AS has_institution,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(year) AS has_year,
    COUNT(type_of_grant) AS has_type_of_grant,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(applicant_name) * 100.0 / COUNT(*), 1) AS pct_applicant,
    ROUND(COUNT(type_of_grant) * 100.0 / COUNT(*), 1) AS pct_type
FROM openalex.awards.carlsberg_fondet_raw;

In [ ]:
%sql
-- Money-shape scan before transform.
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(amount) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    PERCENTILE_APPROX(TRY_CAST(amount AS DOUBLE), 0.5) AS median_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount
FROM openalex.awards.carlsberg_fondet_raw
GROUP BY currency
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution.
SELECT
    TRY_CAST(year AS INT) AS award_year,
    COUNT(*) AS rows,
    COUNT(amount) AS has_amount
FROM openalex.awards.carlsberg_fondet_raw
GROUP BY TRY_CAST(year AS INT)
ORDER BY award_year;

In [ ]:
%sql
-- Grant-type distribution.
SELECT type_of_grant, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
       MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount
FROM openalex.awards.carlsberg_fondet_raw
GROUP BY type_of_grant
ORDER BY rows DESC, type_of_grant;

## Step 1.6: Fail-fast - verify Carlsbergfondet funder row exists

The Step 2 transform cross joins against `openalex.common.funder`. If this funder row is absent, the transform would silently emit zero awards. This assertion must pass before Step 2 runs.

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Carlsbergfondet F4320321504'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320321504;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321504;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carlsberg_fondet_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321504  -- Carlsbergfondet
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.type_of_grant AS funder_scheme,
    'carlsberg_fondet' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) AS end_year,
    CASE
        WHEN n.applicant_name IS NULL OR n.applicant_name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.institution AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.carlsberg_fondet_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.display_name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 121

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'carlsberg_fondet' AND priority = 121;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    121 AS priority  -- Carlsbergfondet priority; 106-120 reserved for open PR backlog
FROM openalex.awards.carlsberg_fondet_awards;

## Step 6: Verification

Expected after admin upload/run: 3,163 rows from the official sitemap/detail-page scrape, `provenance='carlsberg_fondet'`, `priority=121`, DKK amount coverage 100%, applicant/title/year/type coverage 100%, and exactly one Carlsbergfondet funder struct. The script excludes one official sitemap test page (`cf22-0030`, title `TEST FORBINDELSE DRIFT`, amount `DKK 0`).

In [ ]:
%sql
SELECT COUNT(*) AS total_carlsberg_fondet_awards
FROM openalex.awards.carlsberg_fondet_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.carlsberg_fondet_awards;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) AS pct_pi,
    ROUND(COUNT(funder_scheme) * 100.0 / COUNT(*), 1) AS pct_scheme
FROM openalex.awards.carlsberg_fondet_awards;

In [ ]:
%sql
-- Amount and currency sanity. Expect DKK only unless the source expands.
SELECT
    currency,
    COUNT(*) AS rows,
    COUNT(amount) AS has_amount,
    MIN(amount) AS min_amount,
    PERCENTILE_APPROX(amount, 0.5) AS median_amount,
    MAX(amount) AS max_amount
FROM openalex.awards.carlsberg_fondet_awards
GROUP BY currency
ORDER BY rows DESC;

In [ ]:
%sql
-- Year coverage.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.carlsberg_fondet_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Grant-type distribution.
SELECT funder_scheme, COUNT(*) AS rows,
       ROUND(MIN(amount), 0) AS min_amount,
       ROUND(MAX(amount), 0) AS max_amount
FROM openalex.awards.carlsberg_fondet_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;

In [ ]:
%sql
-- Sample for reviewer inspection.
SELECT id,
       SUBSTRING(display_name, 1, 100) AS title,
       amount,
       currency,
       start_year,
       funder_scheme,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       SUBSTRING(lead_investigator.affiliation.name, 1, 80) AS institution,
       landing_page_url
FROM openalex.awards.carlsberg_fondet_awards
ORDER BY start_year DESC, id
LIMIT 20;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.carlsberg_fondet_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Confirm priority insert shape.
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'carlsberg_fondet' AND priority = 121
GROUP BY provenance, priority;